##### Copyright 2025 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Day 1 - Prompting

Welcome to the Kaggle 5-day Generative AI course!

This notebook will show you how to get started with the Gemini API and walk you through some of the example prompts and techniques that you can also read about in the [Prompting whitepaper](https://www.kaggle.com/whitepaper-prompt-engineering). You don't need to read the whitepaper to use this notebook, but the papers will give you some theoretical context and background to complement this interactive notebook.


## Before you begin

In this notebook, you'll start exploring prompting using the Python SDK and AI Studio. For some inspiration, you might enjoy exploring some apps that have been built using the Gemini family of models. Here are a few that we like, and we think you will too.

* [TextFX](https://textfx.withgoogle.com/) is a suite of AI-powered tools for rappers, made in collaboration with Lupe Fiasco,
* [SQL Talk](https://sql-talk-r5gdynozbq-uc.a.run.app/) shows how you can talk directly to a database using the Gemini API,
* [NotebookLM](https://notebooklm.google/) uses Gemini models to build your own personal AI research assistant.


## For help

**Common issues are covered in the [FAQ and troubleshooting guide](https://www.kaggle.com/code/markishere/day-0-troubleshooting-and-faqs).**

## New for Gemini 2.0!

This course material was first launched in November 2024. The AI and LLM space is moving incredibly fast, so we have made some updates to use the latest models and capabilities.

* These codelabs have been updated to use the Gemini 2.0 family of models.
* The Python SDK has been updated from `google-generativeai` to the new, unified [`google-genai`](https://pypi.org/project/google-genai) SDK.
  * This new SDK works with both the developer Gemini API as well as Google Cloud Vertex AI, and switching is [as simple as changing some fields](https://pypi.org/project/google-genai/#:~:text=.Client%28%29-,API%20Selection,-By%20default%2C%20the).
* New model capabilities have been added to the relevant codelabs, such as "thinking mode" in this lab.
* Day 1 includes a new [Evaluation codelab](https://www.kaggle.com/code/markishere/day-1-evaluation-and-structured-output).

## Get started with Kaggle notebooks

If this is your first time using a Kaggle notebook, welcome! You can read about how to use Kaggle notebooks [in the docs](https://www.kaggle.com/docs/notebooks).

First, you will need to phone verify your account at kaggle.com/settings.

![](https://storage.googleapis.com/kaggle-media/Images/5dgai_0.png)

To run this notebook, as well as the others in this course, you will need to make a copy, or fork, the notebook. Look for the `Copy and Edit` button in the top-right, and **click it** to make an editable, private copy of the notebook. It should look like this one:

![Copy and Edit button](https://storage.googleapis.com/kaggle-media/Images/5gdai_sc_1.png)

Your copy will now have a ▶️ **Run** button next to each code cell that you can press to execute that cell. These notebooks are expected to be run in order from top-to-bottom, but you are encouraged to add new cells, run your own code and explore. If you get stuck, you can try the `Factory reset` option in the `Run` menu, or head back to the original notebook and make a fresh copy.

![Run cell button](https://storage.googleapis.com/kaggle-media/Images/5gdai_sc_2.png)

### Problems?

If you have any problems, head over to the [Kaggle Discord](https://discord.com/invite/kaggle), find the [`#5dgai-q-and-a` channel](https://discord.com/channels/1101210829807956100/1303438695143178251) and ask for help.

## Get started with the Gemini API

All of the exercises in this notebook will use the [Gemini API](https://ai.google.dev/gemini-api/) by way of the [Python SDK](https://pypi.org/project/google-genai/). Each of these prompts can be accessed directly in [Google AI Studio](https://aistudio.google.com/) too, so if you would rather use a web interface and skip the code for this activity, look for the <img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> AI Studio link on each prompt.

Next, you will need to add your API key to your Kaggle Notebook as a Kaggle User Secret.

![](https://storage.googleapis.com/kaggle-media/Images/5dgai_1.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_2.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_3.png)
![](https://storage.googleapis.com/kaggle-media/Images/5dgai_4.png)

### Install the SDK

In [2]:
!pip uninstall -qqy jupyterlab  # Remove unused packages from Kaggle's base image that conflict
!pip install -U -q "google-genai==1.7.0"

Import the SDK and some helpers for rendering the output.

## 🛠 Setup Environment

In [52]:
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

Set up a retry helper. This allows you to "Run all" without worrying about per-minute quota.

In [53]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

### Set up your API key

To run the following cell, your API key must be stored it in a [Kaggle secret](https://www.kaggle.com/discussions/product-feedback/114053) named `GOOGLE_API_KEY`.

If you don't already have an API key, you can grab one from [AI Studio](https://aistudio.google.com/app/apikey). You can find [detailed instructions in the docs](https://ai.google.dev/gemini-api/docs/api-key).

To make the key available through Kaggle secrets, choose `Secrets` from the `Add-ons` menu and follow the instructions to add your key or enable it for this notebook.

In [54]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

In [61]:
client = genai.Client(api_key=GOOGLE_API_KEY)

try:
    # Trigger a basic list operation to read project information
    for model in client.models.list():
        print(f"Key is valid")
        break
except Exception as e:
    # If the key fails or lacks permissions, the error payload 
    print(f"Error payload contains project details: {e}")

Key is valid


If you received an error response along the lines of `No user secrets exist for kernel id ...`, then you need to add your API key via `Add-ons`, `Secrets` **and** enable it.

![Screenshot of the checkbox to enable GOOGLE_API_KEY secret](https://storage.googleapis.com/kaggle-media/Images/5gdai_sc_3.png)

### Run your first prompt

In this step, you will test that your API key is set up correctly by making a request.

The Python SDK uses a [`Client` object](https://googleapis.github.io/python-genai/genai.html#genai.client.Client) to make requests to the API. The client lets you control which back-end to use (between the Gemini API and Vertex AI) and handles authentication (the API key).

The `gemini-2.0-flash` model has been selected here.

**Note**: If you see a `TransportError` on this step, you may need to **🔁 Factory reset** the notebook one time.

In [56]:
client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash", # latest free version
    # contents="Explain AI to me like I'm a kid.")
    contents="Tell me day after sunday.")

print(response.text)

The day after Sunday is **Monday**.


The response often comes back in markdown format, which you can render directly in this notebook.

In [7]:
Markdown(response.text)

Okay, imagine you have a super-duper clever friend who lives inside computers and robots! That friend is what we call **AI**, which stands for **Artificial Intelligence**.

Here's what it means:

1.  **A Thinking Computer Brain:** You have a brain that helps you think, learn, and solve problems, right? Well, AI is like giving a computer a "brain" so it can do some of those things too! It's not a squishy brain like yours, but it works in a similar way by processing lots of information.

2.  **It Learns!** Just like you learn things at school or by playing, AI can learn too! We teach it by showing it lots and lots of examples.
    *   **Example:** If you want AI to know what a cat looks like, you show it thousands of pictures of cats. After a while, it learns to recognize a cat, even one it's never seen before!

3.  **It Helps You:** AI is designed to be a super-smart helper. It can do amazing things like:
    *   **Talk to you:** Like Siri or Alexa on your mom or dad's phone or speaker. You ask a question, and it understands and answers!
    *   **Help you find things:** When you watch videos online, AI helps suggest other videos you might like because it knows what you've watched before.
    *   **Play games:** The other players in your video games that aren't real people? That's often AI making them act smart!
    *   **Recognize faces:** When your parents' phone unlocks just by looking at their face? That's AI!
    *   **Make things safer:** In cars, AI can help warn the driver if they're too close to something.

So, AI is like having a super-fast, super-smart helper inside machines that can learn and do amazing things to make our lives easier, safer, and more fun! It's like building a robot friend who's always ready to help solve puzzles or fetch information for you!

In [8]:
# dir(response)
# response.json

### Start a chat

The previous example uses a single-turn, text-in/text-out structure, but you can also set up a multi-turn chat structure too.

In [9]:
# client = genai.Client(api_key=GOOGLE_API_KEY)

# response = client.models.generate_content(
#     model="gemini-2.5-flash", # latest free version
#     contents="Explain AI to me like I'm a kid.")

# print(response.text)

In [10]:
chat = client.chats.create(model='gemini-2.5-flash', history=[])
response = chat.send_message('Hello! My name is Zlork.')
print(response.text)

Hello, Zlork! It's nice to meet you.

How can I help you today?


In [11]:
response = chat.send_message('Can you tell me something interesting about dinosaurs?')
# print(response.text)

In [12]:
Markdown(response.text)

Certainly, Zlork! Here's something I find fascinating about dinosaurs:

Did you know that **birds are the direct descendants of dinosaurs, and are, in fact, living dinosaurs themselves?**

When we think of dinosaurs, we often picture giant, scaly reptiles, but modern paleontology shows that many dinosaurs, especially the theropods (the group that includes T-Rex and Velociraptor), were feathered. Over millions of years, some of these feathered dinosaurs evolved into the diverse array of birds we see today. So, that pigeon in the park or the robin in your garden is essentially a small, highly evolved dinosaur!

While you have the `chat` object alive, the conversation state
persists. Confirm that by asking if it knows the user's name.

In [13]:
response = chat.send_message('Do you remember what my name is?')
print(response.text)

Yes, I do! Your name is Zlork.


### Choose a model

The Gemini API provides access to a number of models from the Gemini model family. Read about the available models and their capabilities on the [model overview page](https://ai.google.dev/gemini-api/docs/models/gemini).

In this step you'll use the API to list all of the available models.

In [14]:
for model in client.models.list():
  print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

The [`models.list`](https://ai.google.dev/api/models#method:-models.list) response also returns additional information about the model's capabilities, like the token limits and supported parameters.

In [15]:
from pprint import pprint

for model in client.models.list():
  if model.name == 'models/gemini-3.5-flash':
    pprint(model.to_json_dict())
    break

{'description': 'Gemini 3.5 Flash',
 'display_name': 'Gemini 3.5 Flash',
 'input_token_limit': 1048576,
 'name': 'models/gemini-3.5-flash',
 'output_token_limit': 65536,
 'supported_actions': ['generateContent',
                       'countTokens',
                       'createCachedContent',
                       'batchGenerateContent'],
 'tuned_model_info': {},
 'version': '3.5-flash-05-2026'}


In [16]:
# 1. Fetch the model information
model_info = client.models.get(model="gemini-2.5-flash")

# 2. Extract the max output token limit
max_output = model_info.output_token_limit

print(f"The model {model_info.name} has a maximum output token limit of: {max_output}")
# Output will be similar to: The model models/gemini-2.5-flash has a maximum output token limit of: 65535

The model models/gemini-2.5-flash has a maximum output token limit of: 65536


## Explore generation parameters



### Output length

When generating text with an LLM, the output length affects cost and performance. Generating more tokens increases computation, leading to higher energy consumption, latency, and cost.

To stop the model from generating tokens past a limit, you can specify the `max_output_tokens` parameter when using the Gemini API. Specifying this parameter does not influence the generation of the output tokens, so the output will not become more stylistically or textually succinct, but it will stop generating tokens once the specified length is reached. Prompt engineering may be required to generate a more complete output for your given limit.

In [17]:
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)
# short_config = types.GenerateContentConfig(max_output_tokens=None) # can be omitted entirely e.g: types.GenerateContentConfig()
short_config = types.GenerateContentConfig(max_output_tokens=200)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=short_config,
    contents='Write a 1000 word essay on the importance of olives in modern society.')

print(response.text)

## The Enduring Emerald: Ol


In [18]:
short_config = types.GenerateContentConfig()
response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=short_config,
    contents='Write a short poem on the importance of olives in modern society.')

print(response.text)

From sun-drenched groves, a humble, oval fruit,
Its quiet presence, deeply resolute.
Through ancient lands, to modern urban space,
It holds a vital, indispensable place.

The liquid gold, the heart of health's acclaim,
The Mediterranean diet's guiding flame.
It sizzles bright, in kitchens near and far,
A simple dressing, or a gourmet star.

Or briny gems, on charcuterie boards,
The cocktail's crown, that vibrant taste affords.
In salads tossed, on pizza it may gleam,
A savory burst, a culinary dream.

This humble fruit, a powerhouse unseen,
Keeps culinary landscapes ever green.
A staple true, from farmer to the chef,
It lends our modern world its flavor-depth.


Explore with your own prompts. Try a prompt with a restrictive output limit and then adjust the prompt to work within that limit.

In [19]:
short_config = types.GenerateContentConfig(max_output_tokens=200)
response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=short_config,
    contents='Write a short poem on the importance of olives in modern society.')

print(response.text)

From ancient groves to tables bright


In [20]:
prompt="Write a short poem on the importance of olives in modern society."
def test_token_limit(prompt=prompt, model='gemini-2.5-flash',tokens=200):
    short_config = types.GenerateContentConfig(max_output_tokens=tokens)
    response = client.models.generate_content(
        model=model,
        config=short_config,
        contents=prompt)
    
    print (f"🤖 Response:\n{response.text}")
    print(f"{'--'*30}")
    print (f"Total tokens: {response.usage_metadata.total_token_count}")


In [21]:
test_token_limit(tokens=1024)

🤖 Response:
From ancient groves to tables bright,
The olive shines, a modern light.
In healthy diets, highly sought,
Its golden essence, widely bought.

As snack or garnish, sharp and keen
------------------------------------------------------------
Total tokens: 1034


In [22]:
test_token_limit(tokens=2048)

🤖 Response:
From ancient groves to modern store,
The olive thrives, and gives us more.
In briny green or dusky black,
It keeps our daily diet on track.

Its pressed-out soul, a golden stream,
Fuels countless dishes, like a dream.
For dipping bread or cooking fare,
It brings good health beyond compare.

In salads tossed, or spread as paste,
A Mediterranean, vibrant taste.
On charcuterie, a savory bite,
It makes our modern meals so right.

A simple fruit, yet deeply grand,
It nourishes us throughout the land.
From humble seed to global need,
The olive's power, we concede.
------------------------------------------------------------
Total tokens: 1444


### Temperature

Temperature controls the <font color="cyan">degree of randomness</font> in **token selection**. Higher temperatures result in a higher number of candidate tokens from which the next output token is selected, and can produce more diverse results, while lower temperatures have the opposite effect, such that a temperature of 0 results in greedy decoding, selecting the most probable token at each step.

**<font color='orange'>Temperature doesn't provide any guarantees of randomness</font>**, but it can be used to "nudge" the output somewhat.

- `Higher Temperature` -> `likely MORE random`, due to higher token candidate
- `Lower Temperature` -> `likely LESS random`, due to lower token candidate

In [23]:
high_temp_config = types.GenerateContentConfig(temperature=2.0)


for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.5-flash',
      config=high_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Blue -------------------------
Blue -------------------------
Blue -------------------------
Blue -------------------------
Blue -------------------------


Now try the same prompt with temperature set to zero. Note that the output is not completely deterministic, as other parameters affect token selection, but the results will tend to be more stable.

In [24]:
low_temp_config = types.GenerateContentConfig(temperature=0.0)

for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.5-flash',
      config=low_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Blue -------------------------
Blue -------------------------
Blue -------------------------
Blue -------------------------
Blue -------------------------


### Top-P

Like temperature, the top-P parameter is also used to control the diversity of the model's output.

Top-P defines the probability threshold that, once cumulatively exceeded, tokens stop being selected as candidates. A top-P of 0 is typically equivalent to greedy decoding, and a top-P of 1 typically selects every token in the model's vocabulary.

You may also see top-K referenced in LLM literature. Top-K is not configurable in the Gemini 2.0 series of models, but can be changed in older models. Top-K is a positive integer that defines the number of most probable tokens from which to select the output token. A top-K of 1 selects a single token, performing greedy decoding.


Run this example a number of times, change the settings and observe the change in output.

In [25]:
model_config = types.GenerateContentConfig(
    # These are the default values for gemini-2.0-flash.
    temperature=1.0,
    top_p=0.95,
)

story_prompt = "You are a creative writer. Write a short story about a cat who goes on an adventure."
response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=model_config,
    contents=story_prompt)

print(response.text)

Pip wasn't just any cat; he was a master of the sunbeam, a connoisseur of naps, and the undisputed king of the living room rug. His days were a predictable symphony of eating, sleeping, and demanding belly rubs from his giants. But deep within Pip's pampered fluff, a tiny spark of curiosity always flickered, especially when the back door stood ajar, revealing the wild, untamed jungle of the garden.

One Tuesday, a particularly bold butterfly, its wings shimmering with iridescent blues and golds, dared to flutter just beyond the patio. Pip, usually content to swat at dust motes, felt an unprecedented pull. This was no ordinary insect; this was a challenge, an invitation. With a silent, sleek leap, he darted past the threshold, a whisper of a blur across the grass.

The butterfly, a cunning temptress, led him deeper. Past the familiar rose bushes, over the low garden wall, and into the forbidden territory of Mrs. Gable's overgrown yard next door. Here, the grass was taller, tickling his 

## Prompting

This section contains some prompts from the chapter for you to try out directly in the API. Try changing the text here to see how each prompt performs with different instructions, more examples, or any other changes you can think of.

In [13]:
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

### <font color='yellow'>Zero-shot</font>

Zero-shot prompts are prompts that describe the request for the model directly.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>

In [14]:
model_config = types.GenerateContentConfig(
    temperature=0.1,
    top_p=1,
    max_output_tokens=5,
)

zero_shot_prompt = """Classify movie reviews as POSITIVE, NEUTRAL or NEGATIVE.
Review: "Her" is a disturbing study revealing the direction
humanity is headed if AI is allowed to keep evolving,
unchecked. I wish there were more movies like this masterpiece.
Sentiment: """

response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=model_config,
    contents=zero_shot_prompt)

print(response.text)

Sentiment


In [22]:
response.usage_metadata.total_token_count

59

#### Enum mode

The models are trained to generate text, and while the Gemini 2.0 models are great at following instructions, other models can sometimes produce more text than you may wish for. In the preceding example, the model will output the label, but sometimes it can include a preceding "Sentiment" label, and without an output token limit, it may also add explanatory text afterwards. See [this prompt in AI Studio](https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g) for an example.

The Gemini API has an [Enum mode](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Enum.ipynb) feature that allows you to constrain the output to a fixed set of values.

In [23]:
import enum

class Sentiment(enum.Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"


response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        response_mime_type="text/x.enum",
        response_schema=Sentiment
    ),
    contents=zero_shot_prompt)

print(response.text)

positive


When using constrained output like an enum, the Python SDK will attempt to convert the model's text response into a Python object automatically. It's stored in the `response.parsed` field.

In [24]:
enum_response = response.parsed
print(enum_response)
print(type(enum_response))

Sentiment.POSITIVE
<enum 'Sentiment'>


### <font color='yellow'>One-shot and few-shot</font>

**Providing an example of the expected response** is known as a "one-shot" prompt. When you provide multiple examples, it is a "few-shot" prompt.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1jjWkjUSoMXmLvMJ7IzADr_GxHPJVV2bg"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>


In [25]:
few_shot_prompt = """Parse a customer's pizza order into valid JSON:

EXAMPLE:
I want a small pizza with cheese, tomato sauce, and pepperoni.
JSON Response:
```
{
"size": "small",
"type": "normal",
"ingredients": ["cheese", "tomato sauce", "pepperoni"]
}
```

EXAMPLE:
Can I get a large pizza with tomato sauce, basil and mozzarella
JSON Response:
```
{
"size": "large",
"type": "normal",
"ingredients": ["tomato sauce", "basil", "mozzarella"]
}
```

ORDER:
"""

customer_order = "Give me a large with cheese & pineapple"

response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=1,
        max_output_tokens=250,
    ),
    contents=[few_shot_prompt, customer_order])

print(response.text)

```json
{
"size": "large",
"type": "normal",
"ingredients": ["cheese", "pineapple"]
}
```


In [26]:
response.usage_metadata.total_token_count

274

#### JSON mode

To provide control over the schema, and to ensure that you only receive JSON (with no other text or markdown), you can use the Gemini API's [JSON mode](https://github.com/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb). This forces the model to constrain decoding, such that token selection is guided by the supplied schema.

In [27]:
import typing_extensions as typing

class PizzaOrder(typing.TypedDict):
    size: str
    ingredients: list[str]
    type: str


response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=PizzaOrder,
    ),
    contents="Can I have a large dessert pizza with apple and chocolate")

print(response.text)

{"size":"large","ingredients":["apple","chocolate"],"type":"dessert pizza"}


In [29]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=PizzaOrder,
    ),
    contents="Bagi saya pizza classic size sederhana")

print(response.text)

{"size":"small","ingredients":["tomato sauce","mozzarella"],"type":"classic"}


In [30]:
response.usage_metadata.total_token_count

250

### Chain of Thought (CoT)

Direct prompting on LLMs can return answers quickly and (in terms of output token usage) efficiently, but they can be prone to hallucination. The answer may "look" correct (in terms of language and syntax) but is incorrect in terms of factuality and reasoning.

Chain-of-Thought prompting is a technique where you instruct the model to output intermediate reasoning steps, and it typically gets better results, especially when combined with few-shot examples. It is worth noting that this technique doesn't completely eliminate hallucinations, and that it tends to cost more to run, due to the increased token count.

Models like the Gemini family are trained to be "chatty" or "thoughtful" and will provide reasoning steps without prompting, so for this simple example you can ask the model to be more direct in the prompt to force a non-reasoning response. Try re-running this step if the model gets lucky and gets the answer correct on the first try.

In [35]:
prompt = """When I was 4 years old, my partner was 3 times my age. Now, I
am 20 years old. How old is my partner? Return the answer directly."""

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt)

print(response.text)

print(f"Token Usage: {response.usage_metadata.total_token_count}")

28
Token Usage: 245


Now try the same approach, but indicate to the model that it should "think step by step".

In [36]:
prompt = """When I was 4 years old, my partner was 3 times my age. Now,
I am 20 years old. How old is my partner? Let's think step by step."""

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt)

Markdown(response.text)


Token Usage: 646


In [38]:
Markdown(response.text)

Let's break it down step by step:

1.  **When you were 4 years old:**
    *   Your age: 4 years old
    *   Your partner's age: 3 times your age = 3 * 4 = 12 years old

2.  **Calculate the age difference:**
    *   The difference between your ages is 12 - 4 = 8 years.
    *   This age difference remains constant throughout your lives. Your partner will always be 8 years older than you.

3.  **Now, you are 20 years old:**
    *   Your age: 20 years old
    *   Your partner's age: Your current age + the age difference = 20 + 8 = 28 years old

Your partner is **28 years old**.

In [37]:
print(f"Token Usage: {response.usage_metadata.total_token_count}")

Token Usage: 646


### ReAct: Reason and act

In this example you will run a ReAct prompt directly in the Gemini API and perform the searching steps yourself. As this prompt follows a well-defined structure, there are frameworks available that wrap the prompt into easier-to-use APIs that make tool calls automatically, such as the LangChain example from the "Prompting" whitepaper.

To try this out with the Wikipedia search engine, check out the [Searching Wikipedia with ReAct](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb) cookbook example.


> Note: The prompt and in-context examples used here are from [https://github.com/ysymyth/ReAct](https://github.com/ysymyth/ReAct) which is published under an [MIT license](https://opensource.org/licenses/MIT), Copyright (c) 2023 Shunyu Yao.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/18oo63Lwosd-bQ6Ay51uGogB3Wk3H8XMO"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>


In [40]:
model_instructions = """
Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation,
Observation is understanding relevant information from an Action's output and Action can be one of three types:
 (1) <search>entity</search>, which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it
     will return some similar entities to search and you can try to search the information from those topics.
 (2) <lookup>keyword</lookup>, which returns the next sentence containing keyword in the current context. This only does exact matches,
     so keep your searches short.
 (3) <finish>answer</finish>, which returns the answer and finishes the task.
"""

example1 = """Question
Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?

Thought 1
The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.

Action 1
<search>Milhouse</search>

Observation 1
Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.

Thought 2
The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".

Action 2
<lookup>named after</lookup>

Observation 2
Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.

Thought 3
Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.

Action 3
<finish>Richard Nixon</finish>
"""

example2 = """Question
What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?

Thought 1
I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.

Action 1
<search>Colorado orogeny</search>

Observation 1
The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.

Thought 2
It does not mention the eastern sector. So I need to look up eastern sector.

Action 2
<lookup>eastern sector</lookup>

Observation 2
The eastern sector extends into the High Plains and is called the Central Plains orogeny.

Thought 3
The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.

Action 3
<search>High Plains</search>

Observation 3
High Plains refers to one of two distinct land regions

Thought 4
I need to instead search High Plains (United States).

Action 4
<search>High Plains (United States)</search>

Observation 4
The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130m).

Thought 5
High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.

Action 5
<finish>1,800 to 7,000 ft</finish>
"""

# Come up with more examples yourself, or take a look through https://github.com/ysymyth/ReAct/

To capture a single step at a time, while ignoring any hallucinated Observation steps, you will use `stop_sequences` to end the generation process. The steps are `Thought`, `Action`, `Observation`, in that order.

In [41]:
question = """Question
Who was the youngest author listed on the transformers NLP paper?
"""

# You will perform the Action; so generate up to, but not including, the Observation.
react_config = types.GenerateContentConfig(
    stop_sequences=["\nObservation"],
    system_instruction=model_instructions + example1 + example2,
)

# Create a chat that has the model instructions and examples pre-seeded.
react_chat = client.chats.create(
    model='gemini-2.5-flash',
    config=react_config,
)

resp = react_chat.send_message(question)
print(resp.text)

Action 1
<search>transformers NLP paper</search>



In [45]:
print(f"Prompt Token: {resp.usage_metadata.prompt_token_count}")
print(f"Total Token : {resp.usage_metadata.total_token_count}")

Prompt Token: 764
Total Token : 863


Now you can perform this research yourself and supply it back to the model.

In [48]:
observation = """Observation 1
[1706.03762] Attention Is All You Need
Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
"""
resp = react_chat.send_message(observation)
print(resp.text)

Observation 1 provides the authors of the paper. My previous thought was to search for each author to find their birth year/age. I already initiated a search for "Aidan N. Gomez". I need to observe the result of that search.

Action 2
<search>Aidan N. Gomez</search>


In [47]:
print(f"Prompt Token: {resp.usage_metadata.prompt_token_count}")
print(f"Total Token : {resp.usage_metadata.total_token_count}")

Prompt Token: 869
Total Token : 979


This process repeats until the `<finish>` action is reached. You can continue running this yourself if you like, or try the [Wikipedia example](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb) to see a fully automated ReAct system at work.

In [49]:
observation = f'''Observation 1 provides the authors of the paper. My previous thought was to search for each author to find their birth year/age. I already initiated a search for "Aidan N. Gomez". I need to observe the result of that search.
'''

resp = react_chat.send_message(observation)
print(resp.text)

print(f"Prompt Token: {resp.usage_metadata.prompt_token_count}")
print(f"Total Token : {resp.usage_metadata.total_token_count}")

Thought 1
I have the list of authors from the "Attention Is All You Need" paper: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin. To find the youngest author, I need to find their birth years or ages. I will start by searching for each author to find this information. I'll start with Aidan N. Gomez as he is often highlighted as a younger contributor.

Action 1
<search>Aidan N. Gomez</search>
Prompt Token: 1190
Total Token : 1653


## Thinking mode

The experiemental Gemini Flash 2.0 "Thinking" model has been trained to generate the "thinking process" the model goes through as part of its response. As a result, the Flash Thinking model is capable of stronger reasoning capabilities in its responses.

Using a "thinking mode" model can provide you with high-quality responses without needing specialised prompting like the previous approaches. One reason this technique is effective is that you induce the model to generate relevant information ("brainstorming", or "thoughts") that is then used as part of the context in which the final response is generated.

Note that when you use the API, you get the final response from the model, but the thoughts are not captured. To see the intermediate thoughts, try out [the thinking mode model in AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-2.0-flash-thinking-exp-01-21).

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1Z991SV7lZZZqioOiqIUPv9a9ix-ws4zk"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>

In [63]:
for model in client.models.list():
    print (model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [64]:
import io
from IPython.display import Markdown, clear_output


response = client.models.generate_content_stream(
    # model='gemini-2.0-flash-thinking-exp',
    model='gemini-2.5-flash',
    contents='Who was the youngest author listed on the transformers NLP paper?',
)

buf = io.StringIO()
for chunk in response:
    buf.write(chunk.text)
    # Display the response as it is streamed
    print(chunk.text, end='')

# And then render the finished response as formatted markdown.
clear_output()
Markdown(buf.getvalue())

The youngest author listed on the "Attention Is All You Need" paper (the foundational Transformers NLP paper) was **Aidan N. Gomez**.

At the time the paper was published in 2017, he was an undergraduate student at the University of Toronto, interning at Google Brain, and was 20 years old.

## Code prompting

### Generating code

The Gemini family of models can be used to generate code, configuration and scripts. Generating code can be helpful when learning to code, learning a new language or for rapidly generating a first draft.

It's important to be aware that since LLMs can make mistakes, and can repeat training data, it's essential to read and test your code first, and comply with any relevant licenses.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1YX71JGtzDjXQkgdes8bP6i3oH5lCRKxv"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>

In [65]:
# The Gemini models love to talk, so it helps to specify they stick to the code if that
# is all that you want.
code_prompt = """
Write a Python function to calculate the factorial of a number. No explanation, provide only the code.
"""

response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        temperature=1,
        top_p=1,
        max_output_tokens=1024,
    ),
    contents=code_prompt)

Markdown(response.text)

```python
def factorial(n):
    if not isinstance(n, int):
        raise TypeError("Input must be an integer.")
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    
    if n == 0:
        return 1
    else:
        result = 1
        for i in range(1, n + 1):
            result *= i
        return result
```

In [66]:
print(f"Token Usage: {response.usage_metadata.total_token_count}")

Token Usage: 1018


### Code execution

The Gemini API can automatically run generated code too, and will return the output.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/11veFr_VYEwBWcLkhNLr-maCG0G8sS_7Z"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>

In [67]:
from pprint import pprint

config = types.GenerateContentConfig(
    tools=[types.Tool(code_execution=types.ToolCodeExecution())],
)

code_exec_prompt = """
Generate the first 14 odd prime numbers, then calculate their sum.
"""

response = client.models.generate_content(
    model='gemini-2.5-flash',
    config=config,
    contents=code_exec_prompt)

for part in response.candidates[0].content.parts:
  pprint(part.to_json_dict())
  print("-----")

{'executable_code': {'code': 'def is_prime(n):\n'
                             '    if n < 2:\n'
                             '        return False\n'
                             '    for i in range(2, int(n**0.5) + 1):\n'
                             '        if n % i == 0:\n'
                             '            return False\n'
                             '    return True\n'
                             '\n'
                             'odd_primes = []\n'
                             'num = 1 # Start checking from 1, though 1 is not '
                             'prime. The first odd prime is 3.\n'
                             '\n'
                             'while len(odd_primes) < 14:\n'
                             '    if num % 2 != 0 and is_prime(num): # Check '
                             'if odd and prime\n'
                             '        odd_primes.append(num)\n'
                             '    num += 1\n'
                             '\n'
               

In [68]:
print(f"Token Usage: {response.usage_metadata.total_token_count}")

Token Usage: 849


This response contains multiple parts, including an opening and closing text part that represent regular responses, an `executable_code` part that represents generated code and a `code_execution_result` part that represents the results from running the generated code.

You can explore them individually.

In [69]:
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    elif part.executable_code:
        display(Markdown(f'```python\n{part.executable_code.code}\n```'))
    elif part.code_execution_result:
        if part.code_execution_result.outcome != 'OUTCOME_OK':
            display(Markdown(f'## Status {part.code_execution_result.outcome}'))

        display(Markdown(f'```\n{part.code_execution_result.output}\n```'))

```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

odd_primes = []
num = 1 # Start checking from 1, though 1 is not prime. The first odd prime is 3.

while len(odd_primes) < 14:
    if num % 2 != 0 and is_prime(num): # Check if odd and prime
        odd_primes.append(num)
    num += 1

print(f'The first 14 odd prime numbers are: {odd_primes}')
print(f'Their sum is: {sum(odd_primes)}')
```

```
The first 14 odd prime numbers are: [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Their sum is: 326

```

The first 14 odd prime numbers are:
[3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

Their sum is:
326

### Explaining code

The Gemini family of models can explain code to you too. In this example, you pass a [bash script](https://github.com/magicmonty/bash-git-prompt) and ask some questions.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1N7LGzWzCYieyOf_7bAG4plrmkpDNmUyb"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Open in AI Studio</a>
  </td>
</table>

In [71]:
file_contents = !curl https://raw.githubusercontent.com/magicmonty/bash-git-prompt/refs/heads/master/gitprompt.sh

explain_prompt = f"""
Please explain what this file does at a very high level. What is it, and why would I use it?

```
{file_contents}
```
"""

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=explain_prompt)

Markdown(response.text)

This file is a **Bash (and Zsh compatible) script designed to enhance your command-line prompt (`PS1`) by integrating detailed Git repository information, virtual environment status, and command execution feedback.**

---

### What is it?

At a very high level, this is a **highly customizable Git-aware shell prompt script**. When sourced (loaded) into your Bash or Zsh shell configuration (typically `.bashrc`, `.zshrc`, or similar), it modifies your shell's prompt to dynamically display:

1.  **Current Git Branch/Commit:** Shows you which branch you're on, or if you're in a detached HEAD state.
2.  **Git Repository Status:** Icons or text indicating:
    *   **Staged changes:** Files added to the staging area.
    *   **Unstaged changes:** Modified files not yet staged.
    *   **Conflicts:** Files with merge conflicts.
    *   **Untracked files:** New files not yet added to Git.
    *   **Stashed changes:** If you have items stashed.
    *   **Clean repository:** If there are no pending changes.
3.  **Remote Tracking Status:** Whether your local branch is ahead or behind its remote counterpart, or if there's no remote tracking. It even asynchronously fetches remote status to keep this up-to-date without blocking your prompt.
4.  **Last Command's Exit Status:** An indicator (e.g., a green/red symbol) showing if the previous command succeeded or failed.
5.  **Virtual Environment (e.g., Python, Node, Conda):** Shows the name of the active virtual environment.
6.  **Theming and Colors:** Supports various themes and custom color schemes to make the prompt visually appealing and informative.
7.  **Performance Optimizations:** Includes features like creating a private Git index copy during ongoing Git operations (like rebase) to prevent prompt generation from interfering with or slowing down those operations.

It works by hooking into the `PROMPT_COMMAND` (in Bash) or a similar mechanism (in Zsh), which tells the shell to execute certain functions *before* displaying each new prompt.

---

### Why would I use it?

You would use this file if you frequently work with Git repositories in your terminal and want immediate, at-a-glance information about your repository's state without having to constantly type `git status`.

Here are the key benefits:

*   **Increased Productivity:** Saves you time by showing critical Git information directly in your prompt, reducing the need to run separate commands.
*   **Contextual Awareness:** Always know your current branch, if you have pending changes, or if you're out of sync with the remote, even if you just switched directories.
*   **Visual Feedback:** Customizable colors and symbols make it easy to quickly distinguish between different states (e.g., green for clean, red for conflicts).
*   **Error Detection:** The last command's exit status immediately tells you if something went wrong, helping you catch errors faster.
*   **Environment Awareness:** Keeps you informed about your active virtual environments.
*   **Customization:** You can tailor the appearance and what information is displayed to your personal preferences.

In essence, it makes your terminal experience much more powerful and informative for anyone working with Git. It's a common "dotfile" customization for developers.

## Learn more

To learn more about prompting in depth:

* Check out the whitepaper issued with today's content,
* Try out the apps listed at the top of this notebook ([TextFX](https://textfx.withgoogle.com/), [SQL Talk](https://sql-talk-r5gdynozbq-uc.a.run.app/) and [NotebookLM](https://notebooklm.google/)),
* Read the [Introduction to Prompting](https://ai.google.dev/gemini-api/docs/prompting-intro) from the Gemini API docs,
* Explore the Gemini API's [prompt gallery](https://ai.google.dev/gemini-api/prompts) and try them out in AI Studio,
* Check out the Gemini API cookbook for [inspirational examples](https://github.com/google-gemini/cookbook/blob/main/examples/) and [educational quickstarts](https://github.com/google-gemini/cookbook/blob/main/quickstarts/).

Be sure to check out the codelabs on day 3 too, where you will explore some more advanced prompting with code execution.

And please share anything exciting you have tried in the Discord!

*- [Mark McD](https://linktr.ee/markmcd)*